In [17]:
import pandas as pd
import glob

# Directory containing your CSV files
csv_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/brick_kilns_neurips_2025/a/new_csv/pakistan/"

# Find all *_land_cover_distribution.csv files in the directory
csv_files = glob.glob(f"{csv_dir}*land_cover_distribution.csv")
print(len(csv_files), "files found.")
dfs = []
for file in csv_files:
    df = pd.read_csv(file, on_bad_lines="skip")
    dfs.append(df)

# Concatenate all dataframes
df = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(df)}")
print(df.head())


1 files found.
Total rows: 125048
              filename  Shrubland  Bare / sparse vegetation  \
0  28.7927_71.4851.png       0.06                     99.80   
1  28.5075_71.4175.png      83.00                     17.00   
2  28.4662_71.0351.png       0.40                     99.46   
3  28.7927_70.1561.png       1.17                      4.02   
4  28.5427_71.8351.png      11.51                     88.49   

   Permanent water bodies  Cropland  Tree cover  Grassland  Built-up  
0                    0.14      0.00        0.00       0.00      0.00  
1                    0.00      0.00        0.00       0.00      0.00  
2                    0.00      0.14        0.00       0.00      0.00  
3                    0.86     92.56        0.47       0.76      0.15  
4                    0.00      0.00        0.00       0.00      0.00  


In [18]:
import pandas as pd

# Your DataFrame (df) assumed already loaded

# Land cover columns
land_cover_cols = [
    "Tree cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare / sparse vegetation", "Permanent water bodies", "Herbaceous wetland"
]

# Find rows where any column exceeds 80
mask = (df[land_cover_cols] >99.90).any(axis=1)
result = df.loc[mask, "filename"]

print(result.tolist())
print(len(result))


KeyError: "['Herbaceous wetland'] not in index"

In [20]:
import pandas as pd
# ,"Herbaceous wetland"
# Assuming df is already loaded with columns: filename + land cover columns
land_cover_cols = [
    "Tree cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare / sparse vegetation", "Permanent water bodies"
]

# Find the column with >99.90 for each row
df_result = []

for _, row in df.iterrows():
    for col in land_cover_cols:
        if row[col] > 99.90:
            df_result.append({
                "filename": row["filename"],
                "land_cover": col
            })
            break  # stop at the first match

# Create DataFrame
result_df = pd.DataFrame(df_result)

print(result_df.head())
print(f"Total rows: {len(result_df)}")


              filename                land_cover
0  29.7264_61.0937.png  Bare / sparse vegetation
1  31.7575_74.1351.png                  Cropland
2  28.5040_72.0351.png  Bare / sparse vegetation
3  28.6426_71.8027.png  Bare / sparse vegetation
4  28.6927_69.6675.png  Bare / sparse vegetation
Total rows: 3969


In [21]:
print(len(result_df))

3969


In [22]:
#  "Herbaceous wetland"

In [24]:
import pandas as pd
# , "Herbaceous wetland"
# Your DataFrame (df) assumed already loaded

land_cover_cols = [
    "Tree cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare / sparse vegetation", "Permanent water bodies"
]
column_counts = {}
for col in land_cover_cols:
    count = (df[col] > 99.90).sum()
    column_counts[col] = count

# print("Column-wise count of images with value >99.90:")
for col, count in column_counts.items():
    print(f"{col}: {count}")

total_count = sum(column_counts.values())
print(total_count)

Tree cover: 497
Shrubland: 51
Grassland: 2799
Cropland: 257
Built-up: 23
Bare / sparse vegetation: 213
Permanent water bodies: 129
3969


In [6]:
# import pandas as pd

# # --- 0) Inputs ---
# # Your original df must include: 'filename' + the 8 land cover columns with percentages
# land_cover_cols = [
#     "Tree cover", "Shrubland", "Grassland", "Cropland",
#     "Built-up", "Bare / sparse vegetation", "Permanent water bodies", "Herbaceous wetland"
# ]

# # Requested retained counts (your targets)
# requested_counts = {
#     "Tree cover": 3919,
#     "Shrubland": 1547,
#     "Grassland": 3674,
#     "Cropland": 6486,
#     "Built-up": 3867,
#     "Bare/Sparse Vegetation": 11226,     # note the slash & casing variant
#     "Permanent Water Bodies": 9011,      # casing variant
#     "Herbaceous Wetland": 1338           # casing variant
# }

# # --- 1) Normalize requested class names -> match df column names ---
# # Map your request keys (variants) to the exact df column names
# name_normalization = {
#     "Tree cover": "Tree cover",
#     "Shrubland": "Shrubland",
#     "Grassland": "Grassland",
#     "Cropland": "Cropland",
#     "Built-up": "Built-up",
#     "Bare/Sparse Vegetation": "Bare / sparse vegetation",
#     "Permanent Water Bodies": "Permanent water bodies",
#     "Herbaceous Wetland": "Herbaceous wetland",
# }

# normalized_requested = {name_normalization[k]: v for k, v in requested_counts.items()}

# # --- 2) Derive (filename, land_cover) using >99.90 rule ---
# # If some rows have multiple >99.90, we'll take the first match in the defined order.
# records = []
# for _, row in df.iterrows():
#     for col in land_cover_cols:
#         if row[col] > 99.90:
#             records.append({"filename": row["filename"], "land_cover": col})
#             break

# labels_df = pd.DataFrame(records)

# # Safety: drop duplicates just in case
# labels_df = labels_df.drop_duplicates(subset=["filename"])

# # --- 3) Sample per class to meet targets (reproducible) ---
# keep_parts = []
# rng = 2025  # fixed seed for reproducibility

# for lc, target_n in normalized_requested.items():
#     pool = labels_df[labels_df["land_cover"] == lc]
#     available = len(pool)
#     take = min(target_n, available)

#     # Sample deterministically by filename sort, then head() OR random sample:
#     # Option A: deterministic (lexicographic)
#     # part = pool.sort_values("filename", kind="mergesort").head(take)

#     # Option B: random sampling but reproducible (recommended to avoid bias)
#     if take > 0:
#         part = pool.sample(n=take, replace=False, random_state=rng)
#     else:
#         part = pool.iloc[0:0]

#     keep_parts.append(part)

# filtered_df = pd.concat(keep_parts, ignore_index=True)

# # --- 4) Sanity summary ---
# summary = (
#     filtered_df.groupby("land_cover")["filename"]
#     .nunique()
#     .reindex(land_cover_cols)  # keep a stable order
#     .fillna(0)
#     .astype(int)
# )

# print("Retained per class:")
# print(summary.to_string())
# print("\nTotal retained:", summary.sum())

# # --- 5) Save for downstream use ---
# # CSV with filename + land_cover
# filtered_df.to_csv("retained_negatives_by_land_cover.csv", index=False)
# print('\nSaved: retained_negatives_by_land_cover.csv')

# # If you only need the list of filenames:
# filenames_list = filtered_df["filename"].tolist()
# print(f"\nUnique filenames retained: {len(set(filenames_list))}")


In [ ]:
import os
import shutil
from pathlib import Path

# --- Inputs ---
# 1) Series (or list) of filenames to copy
#    If you have a pandas Series named `result`, convert it to a list:
#    targets = result.tolist()
targets = [fn.strip() for fn in result.tolist()]  # keep as-is; may or may not include ".png"

# 2) Destination directory
dst_dir = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/bangladesh_negative_samples")
dst_dir.mkdir(parents=True, exist_ok=True)

# 3) Roots to search (local states + NAS Bangladesh RGB)
roots = [
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sindh/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/gujarat/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/west_bengal/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/uttar_pradesh/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/jharkhand/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/rajasthan/rgb"),
    # Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/afghanistan/rgb"),

    # NAS Bangladesh RGB folder
    Path("/mnt/nas_ramanujan/rishabh_sat/sentinel_patch_128_128/pak_punjab/rgb"),
    # Path("/mnt/nas_ramanujan/rishabh_sat/sentinel_patch_128_128/assam/rgb"),
    # Path("/mnt/nas_ramanujan/rishabh_sat/sentinel_patch_128_128/haryana/rgb"),
    # Path("/mnt/nas_ramanujan/rishabh_sat/sentinel_patch_128_128/bihar/rgb"),
]

# --- Helpers ---
def ensure_png_name(name: str) -> str:
    """Attach .png if missing (case-insensitive)."""
    p = Path(name)
    return str(p if p.suffix.lower() == ".png" else p.with_suffix(".png"))

def find_in_roots(filename_png: str):
    """Return the first existing path matching filename_png under any root, else None."""
    for root in roots:
        candidate = root / filename_png
        if candidate.is_file():
            return candidate
    return None

# --- Copy loop ---
copied = 0
skipped_existing = 0
not_found = []

for raw_name in targets:
    filename_png = ensure_png_name(raw_name)
    src_path = find_in_roots(filename_png)

    if src_path is None:
        not_found.append(filename_png)
        continue

    dst_path = dst_dir / Path(filename_png).name

    # Skip if already present (exact name)
    if dst_path.exists():
        skipped_existing += 1
        continue

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, dst_path)
    copied += 1

# --- Report ---
print(f"Copy complete → {dst_dir}")
print(f"Copied: {copied}")
print(f"Already existed (skipped): {skipped_existing}")
print(f"Not found: {len(not_found)}")

if not_found:
    # Print a few missing examples for quick debugging
    sample = "\n".join(not_found[:20])
    print("Missing examples (up to 20 shown):")
    print(sample)


Copy complete → /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/india_negetive_samples
Copied: 37325
Already existed (skipped): 0
Not found: 0


In [27]:
#!/usr/bin/env python3
import os
from pathlib import Path
import shutil

# -------- CONFIG --------
SRC_AFG = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/afganisthan_negative_samples")
SRC_BGD = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/bangladesh_negative_samples")
SRC_IND = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/india_negetive_samples")
SRC_PAK = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/pak_punjab_negative_samples")

DST_ROOT = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_samples")
os.makedirs(DST_ROOT, exist_ok=True)
REQ_AFG = 900
REQ_IND = 30650

# If Bangladesh has a nested 'images' folder, use it; otherwise the given folder
BGD_IMAGES_SUBDIR = "images"

# File extensions to consider as images
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# -------- HELPERS --------
def list_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])

def safe_copy(src: Path, dst_dir: Path) -> Path:
    dst_dir.mkdir(parents=True, exist_ok=True)
    target = dst_dir / src.name
    if not target.exists():
        shutil.copy2(src, target)
        return target
    stem, suf = src.stem, src.suffix
    i = 1
    while True:
        cand = dst_dir / f"{stem}_{i}{suf}"
        if not cand.exists():
            shutil.copy2(src, cand)
            return cand
        i += 1

def copy_limit(src_files: list[Path], limit: int, dst_dir: Path) -> int:
    n = min(limit, len(src_files))
    for p in src_files[:n]:
        safe_copy(p, dst_dir)
    return n

def copy_all(src_files: list[Path], dst_dir: Path) -> int:
    cnt = 0
    for p in src_files:
        safe_copy(p, dst_dir)
        cnt += 1
    return cnt

def count_images(root: Path) -> int:
    return sum(1 for p in root.glob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

# -------- PIPELINE --------
def main():
    # Resolve Bangladesh source
    src_bgd = SRC_BGD / BGD_IMAGES_SUBDIR if (SRC_BGD / BGD_IMAGES_SUBDIR).exists() else SRC_BGD

    # Gather sources
    imgs_afg = list_images(SRC_AFG)
    imgs_bgd = list_images(src_bgd)
    imgs_ind = list_images(SRC_IND)
    imgs_pak = list_images(SRC_PAK)

    # Report availability
    print("Available images")
    print(f"Afghanistan source          {len(imgs_afg)}")
    print(f"Bangladesh source           {len(imgs_bgd)}  (from {src_bgd})")
    print(f"India source                {len(imgs_ind)}")
    print(f"Pakistan source             {len(imgs_pak)}")

    # Prepare destinations
    dst_afg = DST_ROOT / "afghanistan"
    dst_bgd = DST_ROOT / "bangladesh"
    dst_ind = DST_ROOT / "india"
    dst_pak = DST_ROOT / "pakistan"

    # Copy with limits
    copied_afg = copy_limit(imgs_afg, REQ_AFG, dst_afg)
    copied_bgd = copy_all(imgs_bgd, dst_bgd)
    copied_ind = copy_limit(imgs_ind, REQ_IND, dst_ind)
    copied_pak = copy_all(imgs_pak, dst_pak)

    # Sanity check counts
    chk_afg = count_images(dst_afg)
    chk_bgd = count_images(dst_bgd)
    chk_ind = count_images(dst_ind)
    chk_pak = count_images(dst_pak)

    print("\nCopied images")
    print(f"Afghanistan requested {REQ_AFG} copied {copied_afg} present {chk_afg}")
    if copied_afg < REQ_AFG:
        print("Warning Afghanistan source has fewer images than requested")

    print(f"Bangladesh requested all copied {copied_bgd} present {chk_bgd}")

    print(f"India requested {REQ_IND} copied {copied_ind} present {chk_ind}")
    if copied_ind < REQ_IND:
        print("Warning India source has fewer images than requested")

    print(f"Pakistan requested all copied {copied_pak} present {chk_pak}")

    # Global summary
    total_dst = sum(count_images(d) for d in [dst_afg, dst_bgd, dst_ind, dst_pak])
    print(f"\nDestination total images {total_dst} at {DST_ROOT}")

if __name__ == "__main__":
    main()


Available images
Afghanistan source          5898
Bangladesh source           5551  (from /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/bangladesh_negative_samples)
India source                37325
Pakistan source             3969

Copied images
Afghanistan requested 900 copied 900 present 900
Bangladesh requested all copied 5551 present 5551
India requested 30650 copied 30650 present 30650
Pakistan requested all copied 3969 present 3969

Destination total images 41070 at /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_samples


In [28]:
#!/usr/bin/env python3
import random
import shutil
from pathlib import Path

# -------- CONFIG --------
ROOT = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_samples")
OUT_TRAIN = ROOT.parent / "negative_train"
OUT_VAL   = ROOT.parent / "negative_val"
OUT_TEST  = ROOT.parent / "negative_test"

RATIOS = (0.6, 0.2, 0.2)  # train, val, test
SEED = 1337
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# -------- HELPERS --------
def list_images(root: Path) -> list[Path]:
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])

def rel_copy(src: Path, src_root: Path, dst_root: Path):
    rel = src.relative_to(src_root)              # preserve subfolders like afghanistan/, india/, etc.
    target = dst_root / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, target)

def split_indices(n: int, ratios: tuple[float, float, float]) -> tuple[list[int], list[int], list[int]]:
    r_train, r_val, r_test = ratios
    assert abs(r_train + r_val + r_test - 1.0) < 1e-9
    i_train_end = int(n * r_train)
    i_val_end = i_train_end + int(n * r_val)
    idxs = list(range(n))
    random.Random(SEED).shuffle(idxs)
    return idxs[:i_train_end], idxs[i_train_end:i_val_end], idxs[i_val_end:]

# -------- MAIN --------
def main():
    imgs = list_images(ROOT)
    n = len(imgs)
    if n == 0:
        print(f"No images found under {ROOT}")
        return

    train_idx, val_idx, test_idx = split_indices(n, RATIOS)
    print(f"Total {n} | Train {len(train_idx)} Val {len(val_idx)} Test {len(test_idx)}")

    # Copy
    for i in train_idx:
        rel_copy(imgs[i], ROOT, OUT_TRAIN)
    for i in val_idx:
        rel_copy(imgs[i], ROOT, OUT_VAL)
    for i in test_idx:
        rel_copy(imgs[i], ROOT, OUT_TEST)

    # Sanity check counts
    def count(dirp: Path) -> int:
        return sum(1 for p in dirp.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

    c_train = count(OUT_TRAIN)
    c_val   = count(OUT_VAL)
    c_test  = count(OUT_TEST)
    print(f"Written | Train {c_train} Val {c_val} Test {c_test}")
    print(f"Train dir: {OUT_TRAIN}")
    print(f"Val   dir: {OUT_VAL}")
    print(f"Test  dir: {OUT_TEST}")

if __name__ == "__main__":
    main()


Total 41070 | Train 24642 Val 8214 Test 8214
Written | Train 24642 Val 8214 Test 8214
Train dir: /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_train
Val   dir: /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_val
Test  dir: /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_test


In [31]:
#!/usr/bin/env python3
import shutil
from pathlib import Path

SRC = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_test")
DST = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/test/images")

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

def list_images(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])

def safe_copy(src: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    target = dst_dir / src.name
    if not target.exists():
        shutil.copy2(src, target)
    else:
        stem, suf = src.stem, src.suffix
        i = 1
        while True:
            cand = dst_dir / f"{stem}_{i}{suf}"
            if not cand.exists():
                shutil.copy2(src, cand)
                break
            i += 1
def main():
    imgs = list_images(SRC)
    print(f"Found {len(imgs)} images in {SRC}")
    for p in imgs:
        safe_copy(p, DST)
    print(f"Copied images to {DST}")
    print(f"Total in destination: {len(list_images(DST))}")

if __name__ == "__main__":
    main()


Found 8214 images in /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_test
Copied images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/test/images
Total in destination: 18492


In [33]:
## OBB -> DOTA Format
import os
from tqdm import tqdm

# Mapping from class ID to class name
class_mapping = {0: "CFCBK", 1: "FCBK", 2: "Zigzag"}

def convert_to_dota_format(input_path, output_path, image_width, image_height):
    """
    Convert a single label file from normalized format (YOLO-OBB style)
    to DOTA format.
    """
    with open(input_path, "r") as infile, open(output_path, "w") as outfile:
        for line in infile:
            values = line.strip().split()
            if len(values) < 9:
                continue  # Skip incomplete lines

            class_id = int(values[0])
            coordinates = list(map(float, values[1:9]))

            # Denormalize coordinates
            denormalized_coords = [
                round(coordinates[i] * (image_width if i % 2 == 0 else image_height), 2)
                for i in range(8)
            ]

            class_name = class_mapping.get(class_id, "UNKNOWN")
            # Format: x1 y1 x2 y2 x3 y3 x4 y4 class_name difficult
            dota_line = " ".join(map(str, denormalized_coords)) + f" {class_name} 0\n"
            outfile.write(dota_line)

def process_folder(input_folder, output_folder, image_width, image_height):
    """
    Process all `.txt` label files in the input folder and convert to DOTA format.
    """
    if not os.path.exists(input_folder):
        raise FileNotFoundError(f"Input folder '{input_folder}' not found")
    
    os.makedirs(output_folder, exist_ok=True)
    label_files = sorted([f for f in os.listdir(input_folder) if f.endswith(".txt")])

    for file_name in tqdm(label_files, desc="Converting to DOTA format", unit="file"):
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, file_name)
        convert_to_dota_format(input_path, output_path, image_width, image_height)

# === Main Call ===
if __name__ == "__main__":
    input_folder = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/test/yolo_obb_labels"
    output_folder = os.path.join(os.path.dirname(input_folder), "dota_labels")
    image_width = 128
    image_height = 128

    process_folder(input_folder, output_folder, image_width, image_height)

    print(f"\n✅ All label files converted and saved to: {output_folder}")


Converting to DOTA format: 100%|██████████| 10278/10278 [00:00<00:00, 12786.11file/s]


✅ All label files converted and saved to: /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/test/dota_labels


# Negetive Analysis

In [ ]:
import os
import shutil

# Assume `result` is your Series of filenames with any land cover >99.90
# Path to your source images (update if different)
src_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel"
nas="/mnt/nas_ramanujan/rishabh_sat/sentinel_patch_128_128/bangladesh/rgb"

# Destination directory for filtered files
dst_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/sentinelkilndb_bechmarking_data/negative_samples"
os.makedirs(dst_dir, exist_ok=True)

# List of all found rgb folders
rgb_dirs = [
    "sindh/rgb", "gujarat/rgb", "west_bengal/rgb", "uttar_pradesh/rgb",
    "jharkhand/rgb", "rajasthan/rgb", "haryana/rgb",
]

# Copy matching files to new directory
for filename in result.tolist():
    found = False
    for rgb_rel in rgb_dirs:
        rgb_folder = os.path.join(src_dir, rgb_rel)
        src_file = os.path.join(rgb_folder, filename)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, os.path.join(dst_dir, filename))
            found = True
            break
    if not found:
        print(f"File not found: {filename}")

print(f"Copied {len(os.listdir(dst_dir))} files to {dst_dir}")


Copied 41404 files to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/negetive_samples


In [23]:
all_labels = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/images"

In [8]:
import os

In [9]:
all_labels_name = os.listdir(all_labels)

In [10]:
all_labels_name

['31.3074_73.5939.png',
 '25.6451_87.4436.png',
 '26.9555_76.4490.png',
 '25.2791_82.3161.png',
 '26.0716_84.7760.png',
 '25.4791_83.4397.png',
 '26.0716_85.0612.png',
 '29.2292_76.3470.png',
 '26.7208_67.9942.png',
 '24.8540_92.8808.png',
 '30.3074_73.0175.png',
 '23.1848_91.1320.png',
 '30.0251_70.6763.png',
 '26.0879_83.0249.png',
 '27.2883_68.4177.png',
 '22.3673_90.2820.png',
 '30.2426_72.9851.png',
 '31.5339_73.9439.png',
 '24.3463_88.7407.png',
 '26.2468_82.0161.png',
 '25.7997_88.5030.png',
 '34.2600_71.7047.png',
 '22.6652_88.9107.png',
 '22.6848_91.0320.png',
 '27.7791_78.1573.png',
 '28.4968_79.2283.png',
 '26.6304_91.1632.png',
 '29.2644_77.6073.png',
 '22.2959_73.0681.png',
 '26.4128_85.9646.png',
 '28.3083_76.4882.png',
 '28.2967_79.5249.png',
 '23.1459_74.0560.png',
 '23.2415_86.9518.png',
 '26.0791_83.3249.png',
 '31.0838_72.9675.png',
 '22.8261_89.5996.png',
 '26.1716_85.1260.png',
 '26.0379_84.0749.png',
 '26.2967_81.1985.png',
 '29.8041_71.5527.png',
 '25.7879_84.498

In [11]:
result.values

array(['29.2705_75.9794.png', '29.7083_75.4146.png',
       '28.0083_76.9646.png', '28.6617_76.3558.png',
       '29.7969_76.6970.png', '29.4881_76.7382.png',
       '27.9968_76.9680.png', '29.6117_76.2146.png',
       '28.2793_76.5794.png', '29.1469_76.2794.png',
       '29.3116_76.0558.png', '30.4293_77.5180.png',
       '29.6205_76.6882.png', '29.7083_76.8470.png',
       '29.1705_76.8294.png', '29.4469_76.5470.png',
       '30.4381_77.5180.png', '29.9083_74.7058.png',
       '29.7381_76.1882.png', '29.7293_76.1882.png',
       '29.0616_76.5679.png', '29.5968_76.1794.png',
       '29.5968_76.5294.png', '28.4969_76.4180.png',
       '29.6205_75.9882.png', '29.3116_75.9882.png',
       '29.7116_75.0970.png', '29.6793_76.5794.png',
       '29.4469_75.9970.png', '27.9381_77.2680.png',
       '29.5704_74.6147.png', '27.9293_77.2680.png',
       '29.8969_76.6970.png', '29.8793_74.7470.png',
       '29.0616_76.5646.png', '30.4469_77.5146.png',
       '29.7793_75.4382.png', '29.9616_76.1794

In [12]:
import numpy as np

In [13]:
len(np.intersect1d(result.values, all_labels_name))

0

In [ ]:
import pandas as pd
import glob
import os
import shutil

# --- 1. Load all CSVs ---
csv_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/brick_kilns_neurips_2025/a/"
csv_files = glob.glob(f"{csv_dir}*land_cover_distribution.csv")
dfs = [pd.read_csv(file, on_bad_lines="skip") for file in csv_files]
df = pd.concat(dfs, ignore_index=True)

land_cover_cols = [
    "Tree cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare / sparse vegetation", "Permanent water bodies", "Herbaceous wetland"
]

# --- 2. Find all images with >99.90% of ANY land cover class ---
mask_any = (df[land_cover_cols] > 99.93).any(axis=1)
any_result = df.loc[mask_any, "filename"]

print(f"Total images with any class >99.92%: {len(any_result)}")

# --- 3. Per-class lists (for class-wise copying) ---
class_files = {}
for col in land_cover_cols:
    class_mask = (df[col] > 99.9)
    files = df.loc[class_mask, "filename"].tolist()
    class_files[col] = files
    print(f"{col}: {len(files)} images")

# --- 4. Copy to destination: overall and class-wise ---
src_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel"
rgb_dirs = [
    "sindh/rgb", "gujarat/rgb", "delhi_airshed/rgb", "west_bengal/rgb", "wb_small_airshed/rgb",
    "delhi_ncr_region/rgb", "dhaka_airshed/rgb", "khyber_pakhtunkhwa/rgb", "uttar_pradesh/rgb",
    "jharkhand/rgb", "rajasthan/rgb", "afghanistan/rgb", "haryana/rgb", "lucknow_airshed/rgb"
]

# # (A) Copy ALL images with any dominant land cover to a single dir
# dst_dir_all = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/negetive_samples"
# os.makedirs(dst_dir_all, exist_ok=True)

# copied = 0
# for filename in any_result:
#     found = False
#     for rgb_rel in rgb_dirs:
#         src_file = os.path.join(src_dir, rgb_rel, filename)
#         if os.path.isfile(src_file):
#             shutil.copy2(src_file, os.path.join(dst_dir_all, filename))
#             copied += 1
#             found = True
#             break
#     if not found:
#         print(f"File not found: {filename}")
# print(f"Copied {copied} files to {dst_dir_all}")

# (B) Optionally: copy land cover-wise, to subfolders
dst_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise"
for col in land_cover_cols:
    class_dst = os.path.join(dst_root, col.replace(" ", "_"))
    os.makedirs(class_dst, exist_ok=True)
    count = 0
    for filename in class_files[col]:
        found = False
        for rgb_rel in rgb_dirs:
            src_file = os.path.join(src_dir, rgb_rel, filename)
            if os.path.isfile(src_file):
                shutil.copy2(src_file, os.path.join(class_dst, filename))
                count += 1
                found = True
                break
        if not found:
            print(f"File not found: {filename}")
    print(f"Copied {count} images to {class_dst}")


Total images with any class >99.92%: 40558
Tree cover: 3987 images
Shrubland: 1463 images
Grassland: 3683 images
Cropland: 6300 images
Built-up: 3863 images
Bare / sparse vegetation: 11026 images
Permanent water bodies: 8927 images
Herbaceous wetland: 1309 images
Copied 3987 images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise/Tree_cover
Copied 1463 images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise/Shrubland
Copied 3683 images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise/Grassland
Copied 6300 images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise/Cropland
Copied 3863 images to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landco

In [17]:
import os
import pandas as pd
from collections import Counter

# Land cover folders and their names
lc_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise"
land_covers = [
    "Tree_cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare_/_sparse_vegetation", "Permanent_water_bodies", "Herbaceous_wetland"
]

# Model names and their label directories
model_label_dirs = {
    "RT-DETR": "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/predict/processed_labels/negetive_samples_rtdetr/labels",
    "YOLO-Worldv2": "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/predict/processed_labels/negetive_samples_world/labels",
    "YOLOv12L": "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/predict/processed_labels/negetive_samples_yolov12/labels",
    "YOLOv11L-OBB": "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/predict/processed_labels/negetive_samples_yolo-v11-obb/labels"
}
num_classes = 3  # Update if your models use more classes

rows = []
for model, pred_labels_dir in model_label_dirs.items():
    for lc in land_covers:
        lc_dir = os.path.join(lc_root, lc)
        if not os.path.isdir(lc_dir):
            print(f"WARNING: Land cover directory not found: {lc_dir}")
            continue
        png_files = [f for f in os.listdir(lc_dir) if f.endswith('.png')]
        label_basenames = [os.path.splitext(f)[0] for f in png_files]
        
        class_counts = Counter()
        num_images_with_labels = 0
        for base in label_basenames:
            label_path = os.path.join(pred_labels_dir, base + ".txt")
            if os.path.exists(label_path):
                num_images_with_labels += 1
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            try:
                                class_id = int(float(parts[0]))
                                class_counts[class_id] += 1
                            except Exception:
                                pass  # Ignore parse errors
        row = {"Model": model, "Land cover": lc, "Images": len(png_files), "Images with label": num_images_with_labels}
        for c in range(num_classes):
            row[f"Class {c}"] = class_counts[c]
        rows.append(row)

df = pd.DataFrame(rows)

# Print as Markdown table for your report
print(df.to_markdown(index=False))

# Optional: Save as CSV for later use
df.to_csv("landcoverwise_model_class_distribution.csv", index=False)


| Model        | Land cover               |   Images |   Images with label |   Class 0 |   Class 1 |   Class 2 |
|:-------------|:-------------------------|---------:|--------------------:|----------:|----------:|----------:|
| RT-DETR      | Tree_cover               |     4258 |                1790 |        17 |      2097 |       354 |
| RT-DETR      | Shrubland                |     1657 |                 913 |         4 |      1604 |        67 |
| RT-DETR      | Grassland                |     4356 |                2401 |        17 |      3774 |       143 |
| RT-DETR      | Cropland                 |     7524 |                4036 |        36 |      6352 |       304 |
| RT-DETR      | Built-up                 |     4194 |                1991 |        12 |      3169 |       146 |
| RT-DETR      | Bare_/_sparse_vegetation |    11920 |                5703 |        26 |      9296 |       432 |
| RT-DETR      | Permanent_water_bodies   |     9107 |                1991 |        10 |      19

In [18]:
lines = []
for model in df['Model'].unique():
    df_model = df[df['Model'] == model]
    # Add regular rows for this model
    for _, row in df_model.iterrows():
        lines.append(row.tolist())
    # Add horizontal line (markdown) after this model
    lines.append(['---'] * len(df.columns))
    # Add row with sums
    sum_row = [
        model + " (SUM)", "ALL", 
        df_model["Images"].sum(), 
        df_model["Images with label"].sum(),
        df_model["Class 0"].sum(),
        df_model["Class 1"].sum(),
        df_model["Class 2"].sum()
    ]
    lines.append(sum_row)
    # Another markdown line after the sum row
    lines.append(['---'] * len(df.columns))

# Build a DataFrame for pretty output
df_out = pd.DataFrame(lines, columns=df.columns)
print(df_out.to_markdown(index=False))


| Model              | Land cover               | Images   | Images with label   | Class 0   | Class 1   | Class 2   |
|:-------------------|:-------------------------|:---------|:--------------------|:----------|:----------|:----------|
| RT-DETR            | Tree_cover               | 4258     | 1790                | 17        | 2097      | 354       |
| RT-DETR            | Shrubland                | 1657     | 913                 | 4         | 1604      | 67        |
| RT-DETR            | Grassland                | 4356     | 2401                | 17        | 3774      | 143       |
| RT-DETR            | Cropland                 | 7524     | 4036                | 36        | 6352      | 304       |
| RT-DETR            | Built-up                 | 4194     | 1991                | 12        | 3169      | 146       |
| RT-DETR            | Bare_/_sparse_vegetation | 11920    | 5703                | 26        | 9296      | 432       |
| RT-DETR            | Permanent_water_bodies   

In [19]:
land_cover_actual_counts = {
    "Tree_cover": 3919,
    "Shrubland": 1547,
    "Grassland": 3674,
    "Cropland": 6486,
    "Built-up": 3867,
    "Bare_/_sparse_vegetation": 11226,
    "Permanent_water_bodies": 9011,
    "Herbaceous_wetland": 1338
}

# Replace "Images" column for only normal (not SUM) rows:
df.loc[df['Land cover'].isin(land_cover_actual_counts.keys()), 'Images'] = \
    df.loc[df['Land cover'].isin(land_cover_actual_counts.keys()), 'Land cover'].map(land_cover_actual_counts)

# Optionally, replace "ALL" in SUM rows with total:
df.loc[df['Land cover'] == "ALL", 'Images'] = sum(land_cover_actual_counts.values())


In [20]:
print(df.to_markdown(index=False))

| Model        | Land cover               |   Images |   Images with label |   Class 0 |   Class 1 |   Class 2 |
|:-------------|:-------------------------|---------:|--------------------:|----------:|----------:|----------:|
| RT-DETR      | Tree_cover               |     3919 |                1790 |        17 |      2097 |       354 |
| RT-DETR      | Shrubland                |     1547 |                 913 |         4 |      1604 |        67 |
| RT-DETR      | Grassland                |     3674 |                2401 |        17 |      3774 |       143 |
| RT-DETR      | Cropland                 |     6486 |                4036 |        36 |      6352 |       304 |
| RT-DETR      | Built-up                 |     3867 |                1991 |        12 |      3169 |       146 |
| RT-DETR      | Bare_/_sparse_vegetation |    11226 |                5703 |        26 |      9296 |       432 |
| RT-DETR      | Permanent_water_bodies   |     9011 |                1991 |        10 |      19

In [9]:
import os

# Directory with all "negetive_samples" images
neg_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/negetive_samples"
neg_images = set([f for f in os.listdir(neg_dir) if f.endswith('.png')])

# Parent directory containing all landcover subfolders
landcover_parent = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise"
landcover_folders = [os.path.join(landcover_parent, d) for d in os.listdir(landcover_parent) if os.path.isdir(os.path.join(landcover_parent, d))]

landcover_images = set()
for folder in landcover_folders:
    imgs = [f for f in os.listdir(folder) if f.endswith('.png')]
    landcover_images.update(imgs)

# Find images in negetive_samples that are NOT in any landcover folder
extra_images = neg_images - landcover_images

print(f"Total images in negetive_samples: {len(neg_images)}")
print(f"Total unique images in all landcover_wise folders: {len(landcover_images)}")
print(f"Number of extra images in negetive_samples (not in landcover_wise): {len(extra_images)}")
print("Extra images:", list(extra_images)[:10])  # Print first 10 as a sample


Total images in negetive_samples: 41404
Total unique images in all landcover_wise folders: 32444
Number of extra images in negetive_samples (not in landcover_wise): 11282
Extra images: ['22.8749_69.1060.png', '24.0134_69.4560.png', '21.7135_72.4560.png', '23.6871_68.4884.png', '27.5620_69.6589.png', '24.0283_71.0884.png', '21.6547_72.4182.png', '22.6783_72.4795.png', '23.7546_70.0296.png', '24.5871_71.1472.png']


In [10]:
import os
import pandas as pd

neg_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/negetive_samples"
landcover_parent = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/stratified_split/landcover_wise"

neg_images = [f for f in os.listdir(neg_dir) if f.endswith('.png')]

# Get all land cover folders
landcover_folders = [d for d in os.listdir(landcover_parent) if os.path.isdir(os.path.join(landcover_parent, d))]
landcover_paths = {lc: os.path.join(landcover_parent, lc) for lc in landcover_folders}

# Build a lookup: image name → land cover type(s)
img2lc = {}

for img in neg_images:
    found = None
    for lc, lc_dir in landcover_paths.items():
        if os.path.exists(os.path.join(lc_dir, img)):
            found = lc
            break
    img2lc[img] = found  # None if not found in any land cover folder

# Convert to DataFrame for summary
df = pd.DataFrame(list(img2lc.items()), columns=["image", "land_cover"])

# Show stats per land cover
stats = df['land_cover'].value_counts(dropna=False).reset_index()
stats.columns = ["Land cover", "Number of negative samples"]

print("\nPer-land cover count for negative samples:")
print(stats.to_markdown(index=False))

# If you want to save the mapping:
df.to_csv("negative_sample_landcover_mapping.csv", index=False)



Per-land cover count for negative samples:
| Land cover             |   Number of negative samples |
|:-----------------------|-----------------------------:|
|                        |                        11282 |
| Permanent_water_bodies |                         9017 |
| Cropland               |                         6591 |
| Tree_cover             |                         3974 |
| Built-up               |                         3889 |
| Grassland              |                         3745 |
| Shrubland              |                         1565 |
| Herbaceous_wetland     |                         1341 |


In [11]:
print("\nPer-land cover count for negative samples:")
print(stats.to_markdown(index=False))
print(f"\nTotal negative samples: {len(df)}")
print(f"Sum of all counted in land covers (excluding None): {stats[stats['Land cover'].notna()]['Number of negative samples'].sum()}")
print(f"Number of negative samples with no land cover match: {stats[stats['Land cover'].isna()]['Number of negative samples'].sum()}")



Per-land cover count for negative samples:
| Land cover             |   Number of negative samples |
|:-----------------------|-----------------------------:|
|                        |                        11282 |
| Permanent_water_bodies |                         9017 |
| Cropland               |                         6591 |
| Tree_cover             |                         3974 |
| Built-up               |                         3889 |
| Grassland              |                         3745 |
| Shrubland              |                         1565 |
| Herbaceous_wetland     |                         1341 |

Total negative samples: 41404
Sum of all counted in land covers (excluding None): 30122
Number of negative samples with no land cover match: 11282


In [21]:
import pandas as pd

land_cover_actual_counts = {
    "Tree_cover": 3919,
    "Shrubland": 1547,
    "Grassland": 3674,
    "Cropland": 6486,
    "Built-up": 3867,
    "Bare_/_sparse_vegetation": 11226,
    "Permanent_water_bodies": 9011,
    "Herbaceous_wetland": 1338
}
total_actual_images = sum(land_cover_actual_counts.values())

# Replace "Images" for non-sum rows:
df.loc[df['Land cover'].isin(land_cover_actual_counts.keys()), 'Images'] = \
    df.loc[df['Land cover'].isin(land_cover_actual_counts.keys()), 'Land cover'].map(land_cover_actual_counts)

# Optionally, replace "ALL" in SUM rows with actual total:
df.loc[df['Land cover'] == "ALL", 'Images'] = total_actual_images

lines = []
for model in df['Model'].unique():
    df_model = df[df['Model'] == model]
    # Add regular rows for this model
    for _, row in df_model.iterrows():
        lines.append(row.tolist())
    # Add horizontal line (markdown) after this model
    lines.append(['---'] * len(df.columns))
    # Add row with sums: "Images" uses actual total, not sum (to avoid duplication)
    sum_row = [
        model + " (SUM)", "ALL",
        total_actual_images,   # Always use actual sum
        df_model["Images with label"].sum(),
        df_model["Class 0"].sum(),
        df_model["Class 1"].sum(),
        df_model["Class 2"].sum()
    ]
    lines.append(sum_row)
    # Another markdown line after the sum row
    lines.append(['---'] * len(df.columns))

# Build DataFrame for pretty output
df_out = pd.DataFrame(lines, columns=df.columns)
print(df_out.to_markdown(index=False))


| Model              | Land cover               | Images   | Images with label   | Class 0   | Class 1   | Class 2   |
|:-------------------|:-------------------------|:---------|:--------------------|:----------|:----------|:----------|
| RT-DETR            | Tree_cover               | 3919     | 1790                | 17        | 2097      | 354       |
| RT-DETR            | Shrubland                | 1547     | 913                 | 4         | 1604      | 67        |
| RT-DETR            | Grassland                | 3674     | 2401                | 17        | 3774      | 143       |
| RT-DETR            | Cropland                 | 6486     | 4036                | 36        | 6352      | 304       |
| RT-DETR            | Built-up                 | 3867     | 1991                | 12        | 3169      | 146       |
| RT-DETR            | Bare_/_sparse_vegetation | 11226    | 5703                | 26        | 9296      | 432       |
| RT-DETR            | Permanent_water_bodies   

In [22]:
# Suppose your DataFrame is named df and columns are as in your table
landcover_columns = [
    "Tree_cover", "Shrubland", "Grassland", "Cropland",
    "Built-up", "Bare_/_sparse_vegetation", "Permanent_water_bodies", "Herbaceous_wetland"
]

model_list = ['RT-DETR', 'YOLO-Worldv2', 'YOLOv12L', 'YOLOv11L-OBB']

results = []
for model in model_list:
    row = {'Model': model}
    total = 0
    for lc in landcover_columns:
        # Filter the model and landcover, sum all three class columns
        vals = df[(df['Model'] == model) & (df['Land cover'] == lc)]
        detections = vals[['Class 0', 'Class 1', 'Class 2']].sum(axis=1).values
        n_detected = int(detections[0]) if len(detections) else 0
        row[lc] = n_detected
        total += n_detected
    row['TOTAL'] = total
    results.append(row)
df_lc = pd.DataFrame(results)
print(df_lc.to_markdown(index=False))


| Model        |   Tree_cover |   Shrubland |   Grassland |   Cropland |   Built-up |   Bare_/_sparse_vegetation |   Permanent_water_bodies |   Herbaceous_wetland |   TOTAL |
|:-------------|-------------:|------------:|------------:|-----------:|-----------:|---------------------------:|-------------------------:|---------------------:|--------:|
| RT-DETR      |         2468 |        1675 |        3934 |       6692 |       3327 |                       9754 |                     2459 |                  328 |   30637 |
| YOLO-Worldv2 |          427 |         286 |         834 |       1323 |        654 |                       1871 |                      338 |                   46 |    5779 |
| YOLOv12L     |          406 |         229 |         554 |        889 |        473 |                       1461 |                      245 |                   40 |    4297 |
| YOLOv11L-OBB |          402 |         190 |         575 |        931 |        506 |                       1398 |           

In [23]:
# For each land cover column (excluding Model/TOTAL), sum all models
landcover_sums = df_lc[landcover_columns].sum(axis=0)
most_fp_landcover = landcover_sums.idxmax()
print("Land cover class with most detections:", most_fp_landcover)
print("Detections per model in that land cover:")
print(df_lc[['Model', most_fp_landcover]])


Land cover class with most detections: Bare_/_sparse_vegetation
Detections per model in that land cover:
          Model  Bare_/_sparse_vegetation
0       RT-DETR                      9754
1  YOLO-Worldv2                      1871
2      YOLOv12L                      1461
3  YOLOv11L-OBB                      1398


In [27]:
# Filter for the "most_fp_landcover" and each model, and display per-subtype (use your own column names)
subtype_cols = ['Class 0', 'Class 1', 'Class 2']  # replace as needed
for model in model_list:
    print(f"\n{model} in {most_fp_landcover}:")
    vals = df[(df['Model'] == model) & (df['Land cover'] == most_fp_landcover)]
    if not vals.empty:
        print(vals[subtype_cols])



RT-DETR in Bare_/_sparse_vegetation:
   Class 0  Class 1  Class 2
5       26     9296      432

YOLO-Worldv2 in Bare_/_sparse_vegetation:
    Class 0  Class 1  Class 2
13        7     1798       66

YOLOv12L in Bare_/_sparse_vegetation:
    Class 0  Class 1  Class 2
21        9     1375       77

YOLOv11L-OBB in Bare_/_sparse_vegetation:
    Class 0  Class 1  Class 2
29       16     1323       59
